In [2]:
## Imports
from opentrons import protocol_api, simulate, execute
from opentrons.types import Location, Point
import pandas as pd
import numpy as np
from data import data_colordf

In [2]:
##Defining locations - map
x5 = 118.5 + 13.3    ## x5 & y5 are the coordinates for the bottom left corner of plate 5, same idea for x2 & y2 
y5 = 79.3 + 12.3    
x2 = 118.5 + 13.3    ## x_code = x_irl + 13.3
y2 = -11 + 12.3      ## y_code = y_irl + 12.3 

p0 = [[x2 + 0.125*132.4, y2 + 0.75*90.3],#0
      [x2 + 0.375*132.4, y2 + 0.75*90.3],#1
      [x2 + 0.625*132.4, y2 + 0.75*90.3],#2
      [x2 + 0.875*132.4, y2 + 0.75*90.3],#3
      [x2 + 0.125*132.4, y2 + 0.25*90.3],#4
      [x2 + 0.375*132.4, y2 + 0.25*90.3],#5
      [x2 + 0.625*132.4, y2 + 0.25*90.3],#6
      [x2 + 0.875*132.4, y2 + 0.25*90.3]]#7

p1 = [x5+0.25*132.4, y5 + 0.75*90.3]
p2 = [x5 + 0.75*132.4,p1[1]]
p3 = [p1[0],y5 + 0.25*90.3]
p4 = [p2[0],p3[1]]

In [3]:
## The coordinates are absolute 
## with reference to the bottom left corner of slot 1 as origin.
## x, y, and z can be a float or integer

loc0 = [
    Location(Point(p0[0][0],p0[0][1],6),None),
    Location(Point(p0[1][0],p0[1][1],6),None),
    Location(Point(p0[2][0],p0[2][1],6),None),
    Location(Point(p0[3][0],p0[3][1],6),None),
    Location(Point(p0[4][0],p0[4][1],6),None),
    Location(Point(p0[5][0],p0[5][1],6),None),
    Location(Point(p0[6][0],p0[6][1],6),None),
    Location(Point(p0[7][0],p0[7][1],6),None)
]

loc1 = Location(Point(p1[0],p1[1],6),None) # location of blue dye 
loc2 = Location(Point(p2[0],p2[1],6),None) # location of red dye 
loc3 = Location(Point(p3[0],p3[1],6),None) # location of yellow dye
loc4 = Location(Point(p4[0],p4[1],6),None)  # location of green dye


Call protocol to simulate

In [4]:
protocol = simulate.get_protocol_api('2.10')

C:\Users\clara\.opentrons\robot_settings.json not found. Loading defaults
C:\Users\clara\.opentrons\deck_calibration.json not found. Loading defaults


In [5]:
## pipette and labware
tiprack = protocol.load_labware('opentrons_96_tiprack_300ul', '4')
pip = protocol.load_instrument('p300_single_gen2', 'right', tip_racks=[tiprack])

In [6]:
# funtion for moving liquid since repetative
def pipette_action(protocol, pipette, stock_volume_to_pull, stock_position_to_pull, destination_well,stockname):
    pipette.aspirate(stock_volume_to_pull, stock_position_to_pull)
    pipette.dispense(stock_volume_to_pull, destination_well)
    pipette.blow_out()

In [7]:
def stock_in_all(stock,stock_location,tip_max_vol,dis_tray,name):
    '''
    Args- 
    stock: 1D array of all the stock volumes
    stock_location: 1D array of the stock locations corresponding to volumes
    tip_max_vol: float num of the max volume tip can hold
    dis_tray: 1D array of all the locaitons for dispensing
    name: stock name str

    Function-
    Determine the number of times needed to get stock and dispence. 
    If the volume is larger than the tip can hold, automatically determined how many times it has to aspirate and dispence. 
    Care since stock, stock_location, dis_tray are all arrays
    '''
    for i in range(len(stock)):
        stock_amount = stock[i]
        if stock[i] >= tip_max_vol: # eg:250 is max tip volume
            n = stock[i]//tip_max_vol
            for count in range(n):
                pipette_action(protocol, 
                           pipette = pip, 
                           stock_volume_to_pull = tip_max_vol, 
                           stock_position_to_pull = stock_location[i], 
                           destination_well = dis_tray[i],
                           stockname= name )
                stock_amount -= tip_max_vol
                
                if stock_amount <= stock[i] - tip_max_vol*n and stock_amount != 0:
                    pipette_action(protocol, 
                                pipette = pip, 
                                stock_volume_to_pull = (stock[i] - tip_max_vol*n),
                                stock_position_to_pull = stock_location[i], 
                                destination_well = dis_tray[i],
                                stockname= name )
        else: 
            pipette_action(protocol, 
                           pipette = pip, 
                           stock_volume_to_pull = stock[i], 
                           stock_position_to_pull = stock_location[i], 
                           destination_well = dis_tray[i],
                           stockname= name )
        

,mix_loc,stock_red_vol,stock_blue_vol,stock_green_vol,loc_red_stock,loc_blue_stock,loc_green_stock
0,"Location(point=Point(x=148.35000000000002, y=6...",100,550,30,"Location(point=Point(x=231.10000000000002, y=1...","Location(point=Point(x=164.9, y=159.325, z=6),...","Location(point=Point(x=231.10000000000002, y=1..."
1,"Location(point=Point(x=181.45000000000002, y=6...",223,317,20,"Location(point=Point(x=231.10000000000002, y=1...","Location(point=Point(x=164.9, y=159.325, z=6),...","Location(point=Point(x=231.10000000000002, y=1..."
2,"Location(point=Point(x=214.55, y=69.0249999999...",323,130,30,"Location(point=Point(x=231.10000000000002, y=1...","Location(point=Point(x=164.9, y=159.325, z=6),...","Location(point=Point(x=231.10000000000002, y=1..."
3,"Location(point=Point(x=247.65000000000003, y=6...",423,150,10,"Location(point=Point(x=231.10000000000002, y=1...","Location(point=Point(x=164.9, y=159.325, z=6),...","Location(point=Point(x=231.10000000000002, y=1..."
4,"Location(point=Point(x=148.35000000000002, y=2...",523,232,10,"Location(point=Point(x=231.10000000000002, y=1...","Location(point=Point(x=164.9, y=159.325, z=6),...","Location(point=Point(x=231.10000000000002, y=1..."
5,"Location(point=Point(x=181.45000000000002, y=2...",698,423,10,"Location(point=Point(x=231.10000000000002, y=1...","Location(point=Point(x=164.9, y=159.325, z=6),...","Location(point=Point(x=231.10000000000002, y=1..."
6,"Location(point=Point(x=214.55, y=23.875, z=6),...",400,589,10,"Location(point=Point(x=231.10000000000002, y=1...","Location(point=Point(x=164.9, y=159.325, z=6),...","Location(point=Point(x=231.10000000000002, y=1..."
7,"Location(point=Point(x=247.65000000000003, y=2...",309,123,10,"Location(point=Point(x=231.10000000000002, y=1...","Location(point=Point(x=164.9, y=159.325, z=6),...","Location(point=Point(x=231.10000000000002, y=1..."


In [14]:
blue_stock_vol = stocks['stock_blue_vol']
red_stock_vol = stocks['stock_red_vol']
green_stock_vol = stocks['stock_green_vol']

blue_stock_loc = stocks['loc_blue_stock']
red_stock_loc = stocks['loc_red_stock']
green_stock_loc = stocks['loc_green_stock']

dispense = stocks['mix_loc']


In [15]:
def run():
 n=1

In [11]:
## commands
protocol.home()
pip.pick_up_tip()

# move blue
stock_in_all(stock=blue_stock_vol,stock_location=blue_stock_loc,tip_max_vol=300,dis_tray=dispense,name='bluedye')
# move red 
stock_in_all(stock=red_stock_vol,stock_location=red_stock_loc,tip_max_vol=300,dis_tray=dispense,name='bluedye')
# move green
stock_in_all(stock=green_stock_vol,stock_location=green_stock_loc,tip_max_vol=300,dis_tray=dispense,name='bluedye')

pip.drop_tip()

for line in protocol.commands(): 
    print(line)

When aspirate is called on something other than a well relative position, we can't move to the top of the well to prepare for aspiration. This might cause over aspiration if the previous command is a blow_out.
When aspirate is called on something other than a well relative position, we can't move to the top of the well to prepare for aspiration. This might cause over aspiration if the previous command is a blow_out.
When aspirate is called on something other than a well relative position, we can't move to the top of the well to prepare for aspiration. This might cause over aspiration if the previous command is a blow_out.
When aspirate is called on something other than a well relative position, we can't move to the top of the well to prepare for aspiration. This might cause over aspiration if the previous command is a blow_out.
When aspirate is called on something other than a well relative position, we can't move to the top of the well to prepare for aspiration. This might cause over 

Picking up tip from A1 of Opentrons 96 Tip Rack 300 µL on 4
Aspirating 300.0 uL from (164.9, 159.325, 6) at 92.86 uL/sec
Dispensing 300.0 uL into (148.35000000000002, 69.02499999999999, 6) at 92.86 uL/sec
Blowing out at (148.35000000000002, 69.02499999999999, 6)
Aspirating 250.0 uL from (164.9, 159.325, 6) at 92.86 uL/sec
Dispensing 250.0 uL into (148.35000000000002, 69.02499999999999, 6) at 92.86 uL/sec
Blowing out at (148.35000000000002, 69.02499999999999, 6)
Aspirating 300.0 uL from (164.9, 159.325, 6) at 92.86 uL/sec
Dispensing 300.0 uL into (181.45000000000002, 69.02499999999999, 6) at 92.86 uL/sec
Blowing out at (181.45000000000002, 69.02499999999999, 6)
Aspirating 17.0 uL from (164.9, 159.325, 6) at 92.86 uL/sec
Dispensing 17.0 uL into (181.45000000000002, 69.02499999999999, 6) at 92.86 uL/sec
Blowing out at (181.45000000000002, 69.02499999999999, 6)
Aspirating 130.0 uL from (164.9, 159.325, 6) at 92.86 uL/sec
Dispensing 130.0 uL into (214.55, 69.02499999999999, 6) at 92.86 uL/s

In [12]:
#protocol = execute.get_protocol_api('2.7')